In [83]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

import joblib

import warnings
warnings.filterwarnings("ignore")

In [84]:
df = sns.load_dataset("titanic")

In [85]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [86]:
df.isnull().sum()

,0
survived,0
pclass,0
sex,0
age,177
sibsp,0
parch,0
fare,0
embarked,2
class,0
who,0


In [87]:
df.drop("deck",axis=1,inplace=True)

In [88]:
df.isnull().sum()

,0
survived,0
pclass,0
sex,0
age,177
sibsp,0
parch,0
fare,0
embarked,2
class,0
who,0


In [89]:
df['family_size'] = df["sibsp"] + df["parch"] + 1

In [90]:
df["fare_per_person"] = (
    df["fare"] / df["family_size"]
)

In [91]:
df["age_group"] = pd.cut(
    df["age"],
    bins=[0,12,18,35,60,100],
    labels = [
        "Child",
        "Teen",
        "YoungAdult",
        "Adult",
        "Senior"
    ]
)

In [92]:
X = df.drop("survived", axis=1)
y = df["survived"]

In [93]:
numerical_features = df.select_dtypes(include=["int64","float64"]).columns
categorical_features = df.select_dtypes(include=["object","category"]).columns

In [94]:
numerical_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("onehot",OneHotEncoder(handle_unknown="ignore"))
])

In [95]:
X_train, X_test, y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [96]:
numerical_features = X.select_dtypes(include=["int64","float64"]).columns
categorical_features = X.select_dtypes(include=["object","category"]).columns

preprocessing = ColumnTransformer([
    ("num",numerical_pipeline,numerical_features),
    ("cat",categorical_pipeline,categorical_features)
])

pipeline = Pipeline([
    ("preprocessing",preprocessing),
    ("model",RandomForestClassifier())
])

pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['pclass', 'age', 'sibsp', 'parch', 'fare', 'family_size',
       'fare_per_person'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['sex', 'embarked', 'class', 'who', 'embark_town', 'alive', 'age_group'], dtype='object'))])),
                ('model', RandomForestClassifier())])

In [98]:
predictions = pipeline.predict(X_test)

In [99]:
accuracy = accuracy_score(
    y_test,
    predictions
)

print("MODEL ACCURACY:", accuracy)

MODEL ACCURACY: 1.0


In [100]:
report = classification_report(
    y_test,
    predictions
)

print("MODEL REPORT:\n", report)

MODEL REPORT:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       105
           1       1.00      1.00      1.00        74

    accuracy                           1.00       179
   macro avg       1.00      1.00      1.00       179
weighted avg       1.00      1.00      1.00       179

